In [ ]:
# Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 03 · The manual bridge

This is the core lesson. Claude on Vertex has no native `mcp_servers` connector,
so we hand-roll the agentic loop:

1. discover the MCP tools and translate them to Claude's tool schema,
2. send the question to Claude with those tools,
3. when Claude returns a `tool_use` block, call the MCP tool and append the
   `tool_result`,
4. loop until Claude stops requesting tools, then return the final answer.

The loop lives in `src/mcp_claude_bridge.py`; this notebook just runs it and
shows the trace.

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()
sys.path.append(os.path.abspath("src"))

from anthropic import AnthropicVertex
from mcp_client import bq_mcp_session
from mcp_claude_bridge import run_bridge

client = AnthropicVertex(
    project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
    region=os.environ["GOOGLE_CLOUD_REGION"],
)
MODEL_ID = os.environ["CLAUDE_MODEL_ID"]

## Ask a natural-language question against a public dataset

The question targets `bigquery-public-data.usa_names.usa_1910_2013` (an inline
public literal, so the tutorial runs without private data). Claude decides which
MCP tool to call and with what arguments.

In [ ]:
# The execution project (where BigQuery jobs run) comes from .env; the data lives
# in the public bigquery-public-data project. Querying public data still requires
# a billing/execution project, so we tell Claude which one to use.
EXEC_PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]

QUESTION = (
    f"Run BigQuery jobs in project '{EXEC_PROJECT}' using the execute_sql_readonly "
    "tool. Using the public dataset `bigquery-public-data.usa_names.usa_1910_2013`, "
    "which girls' first name (gender = 'F') was registered the most in California "
    "(state = 'CA') in the year 1995? Return the name and the count."
)

async with bq_mcp_session() as session:
    answer, trace = await run_bridge(client, session, MODEL_ID, QUESTION)

## The full trace

Claude's tool request(s) → the MCP result(s) → the final answer.

In [ ]:
for kind, payload in trace:
    if kind == "tool_use":
        print(f"\n[Claude -> MCP] call {payload['name']}")
        print("  input:", payload["input"])
    elif kind == "tool_result":
        flag = " (error)" if payload["is_error"] else ""
        print(f"[MCP -> Claude] result from {payload['name']}{flag}:")
        print("  ", payload["text"][:800])
    elif kind == "final":
        print("\n[Final answer]")
        print(payload)

## Why Vertex needs this, when the Anthropic API does not

On the **Anthropic API** (`api.anthropic.com`), this is one block — you hand the
platform the server and it runs the loop for you:

```python
# Anthropic API only — NOT supported on Vertex
client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    messages=[{"role": "user", "content": QUESTION}],
    mcp_servers=[{"type": "url", "url": MCP_ENDPOINT, "name": "bigquery"}],
)
```

On **Vertex**, the `mcp_servers` parameter is not supported — the Vertex feature
list explicitly excludes the MCP connector and programmatic tool calling. So
*we* are the connector: discover, translate, dispatch, loop. That is exactly
what `run_bridge` does above. Notebook 04 hands this same job to ADK.

## Cleanup

Nothing to clean up: the MCP session is context-managed (`async with`) and closes
automatically, and the bridge creates no persistent cloud resources.